# Paper 1 — Complete TrOCR-LoRA Ablation (One-Shot, Kaggle)

**One run = entire Paper 1 experiment suite:** dataset download → 6 LoRA
configs + full fine-tune (with BOTH bug fixes: preprocessing parity +
corrected double-shift loss) → full test-split evaluation (bootstrap CIs,
akshara error rate) → error analysis → tables → one zip to download.

**Setup:** GPU T4 x2, Internet ON. ~7-9h. **Resumable:** re-Run-All skips finished runs.

In [ ]:
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install "transformers==4.57.6" peft datasets editdistance accelerate
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
try:
    import torchao; raise SystemExit("torchao still present — restart session and rerun this cell")
except ImportError:
    print("torchao removed — peft dispatch safe")

In [ ]:
# ---- Dataset: official HF parquets (pinned revision), resumable download ----
import os
os.makedirs("data/iiit_hindi_parquet", exist_ok=True)
BASE = "https://huggingface.co/datasets/c3rl/IIIT-INDIC-HW-WORDS-Hindi/resolve/2a27244ff5f5f5eaaf86aa4b9411beb356921f51/data"
FILES = ["train-00000-of-00003.parquet","train-00001-of-00003.parquet","train-00002-of-00003.parquet",
         "validation-00000-of-00001.parquet","test-00000-of-00001.parquet"]
for f in FILES:
    !curl -sL --fail --continue-at - --retry 10 --retry-delay 5 -o data/iiit_hindi_parquet/{f} {BASE}/{f}
!ls -la data/iiit_hindi_parquet/

In [ ]:
import os
for d in ["backend", "paper1/experiments"]:
    os.makedirs(d, exist_ok=True)
for f in ["backend/__init__.py", "paper1/__init__.py", "paper1/experiments/__init__.py"]:
    open(f, "w").close()
print("package dirs ready")

In [ ]:
%%writefile backend/preprocessing.py
"""
DevGen Framework — Advanced Preprocessing Pipeline
Handles: Binarization, Deskewing, Denoising, Normalization
"""

import cv2
import numpy as np
from PIL import Image
import io


def bytes_to_cv2(image_bytes: bytes) -> np.ndarray:
    """Convert raw image bytes to OpenCV BGR image."""
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return img


def cv2_to_pil(img: np.ndarray) -> Image.Image:
    """Convert OpenCV BGR image to PIL RGB Image."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def binarize(img: np.ndarray) -> np.ndarray:
    """Adaptive binarization for handwritten documents.
    Uses Gaussian adaptive thresholding to handle uneven lighting."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, blockSize=15, C=10
    )
    return binary


def deskew(img: np.ndarray) -> np.ndarray:
    """Correct document skew using Hough Line Transform."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100,
                            minLineLength=100, maxLineGap=10)
    if lines is None:
        return img

    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(angle) < 45:  # Only consider near-horizontal lines
            angles.append(angle)

    if not angles:
        return img

    median_angle = np.median(angles)
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, median_angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return rotated


def denoise(img: np.ndarray) -> np.ndarray:
    """Non-local means denoising for handwritten documents."""
    if len(img.shape) == 3:
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    else:
        return cv2.fastNlMeansDenoising(img, None, 10, 7, 21)


def normalize_for_model(img: np.ndarray, target_height: int = 384,
                         target_width: int = 384) -> np.ndarray:
    """Resize image to target dimensions while maintaining aspect ratio.
    Pads with white if needed."""
    h, w = img.shape[:2]
    scale = min(target_height / h, target_width / w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Create white canvas and center the image
    if len(img.shape) == 3:
        canvas = np.ones((target_height, target_width, 3), dtype=np.uint8) * 255
    else:
        canvas = np.ones((target_height, target_width), dtype=np.uint8) * 255

    y_offset = (target_height - new_h) // 2
    x_offset = (target_width - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    return canvas


def crop_to_foreground(img: np.ndarray, padding_ratio: float = 0.18) -> np.ndarray:
    """Crop around visible handwriting while keeping a little context."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img

    h, w = gray.shape[:2]
    min_area = max(12, int(h * w * 0.0001))
    boxes = [cv2.boundingRect(contour) for contour in contours if cv2.contourArea(contour) >= min_area]
    if not boxes:
        return img

    x1 = min(x for x, _, _, _ in boxes)
    y1 = min(y for _, y, _, _ in boxes)
    x2 = max(x + bw for x, _, bw, _ in boxes)
    y2 = max(y + bh for _, y, _, bh in boxes)

    pad_x = max(8, int((x2 - x1) * padding_ratio))
    pad_y = max(8, int((y2 - y1) * padding_ratio))
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)

    return img[y1:y2, x1:x2]


def preprocess_cv2_for_ocr(img: np.ndarray) -> np.ndarray:
    """Word-image preprocessing on a BGR array.
    Matches HF Space Logic:
    - AR <= 1.55: Crop to foreground.
    - AR > 2.2: Crop to foreground + Pad to Square.
    - 1.55 < AR <= 2.2: No change (raw).
    """
    h, w = img.shape[:2]
    aspect_ratio = w / float(h)

    if aspect_ratio <= 1.55:
        img = crop_to_foreground(img)
    elif aspect_ratio > 2.2:
        img = crop_to_foreground(img)
        img = normalize_for_model(img, target_height=384, target_width=384)
    return img


def preprocess_for_ocr(image_bytes: bytes) -> Image.Image:
    """Prepare a word image for TrOCR from raw bytes (serving path)."""
    img = bytes_to_cv2(image_bytes)
    if img is None: return None
    return cv2_to_pil(preprocess_cv2_for_ocr(img))


def preprocess_pil_for_ocr(image: Image.Image) -> Image.Image:
    """Prepare a PIL word image for TrOCR without a bytes round-trip
    (training/eval path — must stay in parity with preprocess_for_ocr)."""
    img = cv2.cvtColor(np.array(image.convert("RGB")), cv2.COLOR_RGB2BGR)
    return cv2_to_pil(preprocess_cv2_for_ocr(img))


def full_preprocess(image_bytes: bytes) -> Image.Image:
    """Complete preprocessing pipeline for document images.
    Steps: Denoise → Deskew → Binarize → Normalize → PIL"""
    img = bytes_to_cv2(image_bytes)
    img = denoise(img)
    img = deskew(img)
    # Keep color for ViT input (TrOCR expects RGB images)
    img = normalize_for_model(img)
    return cv2_to_pil(img)


In [ ]:
%%writefile backend/train_trocr.py
"""
Fine-tune TrOCR for Devanagari handwriting with LoRA.

This script is Mac/Apple Silicon friendly by default:
- uses MPS when available
- avoids CUDA-only fp16 training on MPS
- can train directly from HuggingFace or from extracted ./data folders
"""

from __future__ import annotations

import argparse
import io
import json
import os
import unicodedata
from pathlib import Path
from typing import Any, Optional

import editdistance
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image
from torch.utils.data import ConcatDataset, Dataset
from transformers import (
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrOCRProcessor,
    ViTImageProcessor,
    VisionEncoderDecoderModel,
    default_data_collator,
    set_seed,
)

MODEL_NAME = "paudelanil/trocr-devanagari-2"
DATASET_NAME = "c3rl/IIIT-INDIC-HW-WORDS-Hindi"
DEFAULT_IMAGE_PROCESSOR = "google/vit-base-patch16-224-in21k"

# LoRA target-module presets. PEFT treats a string as a regex over full module
# paths and a list as suffix matches. "legacy" reproduces the shipped adapter
# (note: bare "dense" also matches ViT FFN layers via suffix matching).
TARGET_MODULE_PRESETS: dict[str, Any] = {
    "legacy": ["query", "key", "value", "dense"],
    "attn": r".*(query|key|value|q_proj|k_proj|v_proj|out_proj|attention\.output\.dense)$",
    "attn-ffn": r".*(query|key|value|q_proj|k_proj|v_proj|out_proj|dense|fc1|fc2)$",
}


def get_best_torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_trocr_processor(model_name: str) -> TrOCRProcessor:
    try:
        return TrOCRProcessor.from_pretrained(model_name)
    except Exception as exc:
        print(f"Falling back to ViT image processor + checkpoint tokenizer: {exc}")
        image_processor = ViTImageProcessor.from_pretrained(DEFAULT_IMAGE_PROCESSOR)
        image_processor.image_mean = [0.5, 0.5, 0.5]
        image_processor.image_std = [0.5, 0.5, 0.5]
        image_processor.rescale_factor = 1.0 / 255.0
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        return TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)


class ExtractedDevanagariDataset(Dataset):
    def __init__(
        self,
        split_dir: Path,
        processor: TrOCRProcessor,
        max_target_length: int,
        limit: Optional[int] = None,
        preprocess: str = "none",
    ):
        self.split_dir = split_dir
        self.processor = processor
        self.max_target_length = max_target_length
        self.preprocess = preprocess
        self.df = pd.read_csv(split_dir / "labels.csv")
        if limit:
            self.df = self.df.head(limit)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        image_path = self.split_dir / "images" / row["filename"]
        image = Image.open(image_path).convert("RGB")
        return encode_sample(image, str(row["text"]), self.processor, self.max_target_length, self.preprocess)


class HuggingFaceDevanagariDataset(Dataset):
    def __init__(
        self,
        split: str,
        processor: TrOCRProcessor,
        max_target_length: int,
        limit: Optional[int] = None,
        preprocess: str = "none",
    ):
        split_spec = f"{split}[:{limit}]" if limit else split
        self.dataset = load_dataset(DATASET_NAME, split=split_spec)
        self.processor = processor
        self.max_target_length = max_target_length
        self.preprocess = preprocess

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        sample = self.dataset[idx]
        image = coerce_to_pil(sample["image"])
        return encode_sample(image, str(sample["text"]), self.processor, self.max_target_length, self.preprocess)


class ParquetDevanagariDataset(Dataset):
    def __init__(
        self,
        parquet_dir: Path,
        split: str,
        processor: TrOCRProcessor,
        max_target_length: int,
        limit: Optional[int] = None,
        preprocess: str = "none",
    ):
        pattern = parquet_dir / f"{split}-*.parquet"
        files = sorted(str(path) for path in parquet_dir.glob(f"{split}-*.parquet"))
        if not files:
            raise FileNotFoundError(f"No parquet files found for {split!r}: {pattern}")
        split_spec = f"train[:{limit}]" if limit else "train"
        self.dataset = load_dataset("parquet", data_files=files, split=split_spec)
        self.processor = processor
        self.max_target_length = max_target_length
        self.preprocess = preprocess

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        sample = self.dataset[idx]
        image = coerce_to_pil(sample["image"])
        return encode_sample(image, str(sample["text"]), self.processor, self.max_target_length, self.preprocess)


class TrOCRSeq2SeqTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer with PEFT saving adjusted for VisionEncoderDecoder configs.

    Also overrides compute_loss: transformers 4.57's VisionEncoderDecoder
    routes `labels` through ForCausalLMLoss, which shifts targets a second
    time after decoder_input_ids were already shifted from labels. The
    double shift trains the model to predict the PREVIOUS token (base-model
    teacher-forced loss reads 15.75 where correct alignment reads 0.45) and
    collapses generation. We build decoder_input_ids ourselves and compute
    cross-entropy against unshifted labels.
    """

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        cfg_model = model
        while hasattr(cfg_model, "module"):  # unwrap DataParallel/DDP
            cfg_model = cfg_model.module
        config = cfg_model.config  # PeftModel forwards .config to the base model
        pad_id = config.pad_token_id
        start_id = config.decoder_start_token_id

        decoder_input_ids = labels.new_full(labels.shape, pad_id)
        decoder_input_ids[:, 1:] = labels[:, :-1].clone()
        decoder_input_ids[:, 0] = start_id
        decoder_input_ids[decoder_input_ids == -100] = pad_id

        outputs = model(
            pixel_values=inputs["pixel_values"],
            decoder_input_ids=decoder_input_ids,
        )
        logits = outputs.logits
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
        )
        return (loss, outputs) if return_outputs else loss

    def _save(self, output_dir: Optional[str] = None, state_dict: Optional[dict] = None) -> None:
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        if state_dict is None:
            state_dict = self.model.state_dict()

        if isinstance(self.model, PeftModel):
            self.model.save_pretrained(
                output_dir,
                state_dict=state_dict,
                save_embedding_layers=False,
            )
        elif hasattr(self.model, "save_pretrained"):
            self.model.save_pretrained(output_dir, state_dict=state_dict)
        else:
            torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))

        if self.processing_class is not None:
            self.processing_class.save_pretrained(output_dir)

        torch.save(self.args, os.path.join(output_dir, "training_args.bin"))


def coerce_to_pil(image_data: Any) -> Image.Image:
    if isinstance(image_data, Image.Image):
        return image_data.convert("RGB")
    if isinstance(image_data, dict) and "bytes" in image_data:
        return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
    if isinstance(image_data, bytes):
        return Image.open(io.BytesIO(image_data)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image_data)!r}")


def encode_sample(
    image: Image.Image,
    text: str,
    processor: TrOCRProcessor,
    max_target_length: int,
    preprocess: str = "none",
) -> dict[str, torch.Tensor]:
    if preprocess == "app":
        # Crop/pad-to-square parity with the base checkpoint's training regime
        # (see backend/preprocessing.py). Raw wide strips mismatch the base
        # model's input distribution and destroy matra recognition.
        try:
            from backend.preprocessing import preprocess_pil_for_ocr
        except ImportError:
            from preprocessing import preprocess_pil_for_ocr
        image = preprocess_pil_for_ocr(image)
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
    labels = processor.tokenizer(
        text,
        padding="max_length",
        max_length=max_target_length,
        truncation=True,
    ).input_ids
    labels = [
        label if label != processor.tokenizer.pad_token_id else -100
        for label in labels
    ]
    return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


def build_datasets(args: argparse.Namespace, processor: TrOCRProcessor):
    if args.data_dir:
        data_dir = Path(args.data_dir).expanduser().resolve()
        train_dataset = ExtractedDevanagariDataset(
            data_dir / "train",
            processor,
            args.max_target_length,
            limit=args.train_limit,
            preprocess=args.preprocess,
        )
        eval_dataset = ExtractedDevanagariDataset(
            data_dir / "validation",
            processor,
            args.max_target_length,
            limit=args.eval_limit,
            preprocess=args.preprocess,
        )
    elif args.parquet_dir:
        parquet_dir = Path(args.parquet_dir).expanduser().resolve()
        train_dataset = ParquetDevanagariDataset(
            parquet_dir,
            "train",
            processor,
            args.max_target_length,
            limit=args.train_limit,
            preprocess=args.preprocess,
        )
        eval_dataset = ParquetDevanagariDataset(
            parquet_dir,
            "validation",
            processor,
            args.max_target_length,
            limit=args.eval_limit,
            preprocess=args.preprocess,
        )
    else:
        train_dataset = HuggingFaceDevanagariDataset(
            "train",
            processor,
            args.max_target_length,
            limit=args.train_limit,
            preprocess=args.preprocess,
        )
        eval_dataset = HuggingFaceDevanagariDataset(
            "validation",
            processor,
            args.max_target_length,
            limit=args.eval_limit,
            preprocess=args.preprocess,
        )

    if args.synthetic_dir:
        synthetic_dataset = ExtractedDevanagariDataset(
            Path(args.synthetic_dir).expanduser().resolve(),
            processor,
            args.max_target_length,
            limit=args.synthetic_limit,
            preprocess=args.preprocess,
        )
        print(f"Mixing {len(synthetic_dataset)} synthetic samples into {len(train_dataset)} real ones")
        train_dataset = ConcatDataset([train_dataset, synthetic_dataset])

    return train_dataset, eval_dataset


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Fine-tune TrOCR LoRA for Devanagari OCR.")
    parser.add_argument("--base-model", default=os.getenv("TROCR_BASE_MODEL", MODEL_NAME))
    parser.add_argument("--output-dir", default=os.getenv("TROCR_ADAPTER_ROOT", "./trocr-devanagari-lora"))
    parser.add_argument("--data-dir", default=None, help="Optional extracted dataset directory containing train/validation.")
    parser.add_argument("--parquet-dir", default=None, help="Optional local HF parquet directory containing train/validation/test parquet files.")
    parser.add_argument("--epochs", type=float, default=3.0)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--eval-batch-size", type=int, default=4)
    parser.add_argument("--grad-accum", type=int, default=2)
    parser.add_argument("--learning-rate", type=float, default=5e-5)
    parser.add_argument("--max-target-length", type=int, default=128)
    parser.add_argument("--generation-max-length", type=int, default=None,
                        help="Max generated tokens during validation/test; defaults to model generation config.")
    parser.add_argument("--eval-steps", type=int, default=1000)
    parser.add_argument("--save-steps", type=int, default=1000)
    parser.add_argument("--eval-strategy", choices=["steps", "epoch", "no"], default="steps")
    parser.add_argument("--save-strategy", choices=["steps", "epoch", "no"], default="steps")
    parser.add_argument("--logging-steps", type=int, default=50)
    parser.add_argument("--train-limit", type=int, default=None, help="Use a small subset for smoke tests.")
    parser.add_argument("--eval-limit", type=int, default=None, help="Use a small validation subset for smoke tests.")
    parser.add_argument("--preprocess", choices=["none", "app"], default="app",
                        help="app = crop/pad-to-square parity with the base checkpoint (recommended); "
                             "none = raw dataset images.")
    parser.add_argument("--num-workers", type=int, default=0, help="Dataloader workers (preprocessing is CPU-bound).")
    parser.add_argument("--device", choices=["auto", "cpu", "mps", "cuda"], default="auto")
    parser.add_argument("--seed", type=int, default=42)
    # LoRA ablation knobs (Paper 1)
    parser.add_argument("--lora-r", type=int, default=16)
    parser.add_argument("--lora-alpha", type=int, default=None, help="Defaults to 2 * lora_r.")
    parser.add_argument("--lora-dropout", type=float, default=0.05)
    parser.add_argument(
        "--target-modules",
        choices=sorted(TARGET_MODULE_PRESETS),
        default="legacy",
        help="legacy = shipped-adapter config; attn = attention projections only; attn-ffn = attention + FFN.",
    )
    parser.add_argument("--full-finetune", action="store_true", help="Train all weights (no LoRA baseline).")
    # Synthetic data augmentation (Paper 1)
    parser.add_argument("--synthetic-dir", default=None,
                        help="Directory with images/ + labels.csv from generate_synthetic.py, mixed into training.")
    parser.add_argument("--synthetic-limit", type=int, default=None,
                        help="Cap synthetic samples (for 10%%/25%%/50%% mixing ratios).")
    parser.add_argument("--eval-test", action="store_true",
                        help="After training, evaluate on the official test split and write test_metrics.json.")
    parser.add_argument("--resume-from-checkpoint", default=None,
                        help="Resume Trainer state from a checkpoint directory, e.g. output/checkpoint-4366.")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    device = get_best_torch_device() if args.device == "auto" else args.device
    if device == "mps":
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

    print(f"Initializing TrOCR {'full fine-tune' if args.full_finetune else 'LoRA'} training on {device}")
    processor = load_trocr_processor(args.base_model)
    model = VisionEncoderDecoderModel.from_pretrained(args.base_model)

    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id
    generation_max_length = args.generation_max_length or model.generation_config.max_length or args.max_target_length
    model.generation_config.max_length = generation_max_length
    model.generation_config.num_beams = 4

    if not args.full_finetune:
        lora_config = LoraConfig(
            r=args.lora_r,
            lora_alpha=args.lora_alpha if args.lora_alpha is not None else 2 * args.lora_r,
            lora_dropout=args.lora_dropout,
            target_modules=TARGET_MODULE_PRESETS[args.target_modules],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} / {total_params:,}")

    train_dataset, eval_dataset = build_datasets(args, processor)
    print(f"Train samples: {len(train_dataset)}")
    print(f"Validation samples: {len(eval_dataset)}")

    def compute_metrics(pred):
        labels_ids = pred.label_ids.copy()
        pred_ids = pred.predictions
        labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
        labels_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

        cer_sum = 0.0
        word_errors = 0
        valid_samples = 0
        for prediction, label in zip(pred_str, labels_str):
            prediction = unicodedata.normalize("NFC", prediction.strip())
            label = unicodedata.normalize("NFC", label.strip())
            if label:
                cer_sum += editdistance.eval(prediction, label) / len(label)
                word_errors += int(prediction != label)
                valid_samples += 1

        if not valid_samples:
            return {"cer": 0.0, "wer": 0.0}
        return {"cer": cer_sum / valid_samples, "wer": word_errors / valid_samples}

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        predict_with_generate=True,
        eval_strategy=args.eval_strategy,
        save_strategy=args.save_strategy,
        eval_steps=args.eval_steps,
        save_steps=args.save_steps,
        logging_steps=args.logging_steps,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.learning_rate,
        num_train_epochs=args.epochs,
        remove_unused_columns=False,
        fp16=device == "cuda",
        dataloader_pin_memory=device == "cuda",
        dataloader_num_workers=args.num_workers,
        save_total_limit=3,
        load_best_model_at_end=args.eval_strategy != "no" and args.save_strategy != "no",
        metric_for_best_model="cer",
        greater_is_better=False,
        generation_max_length=generation_max_length,
        report_to="none",
    )

    trainer = TrOCRSeq2SeqTrainer(
        model=model,
        processing_class=processor.image_processor,
        args=training_args,
        compute_metrics=compute_metrics,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator,
    )

    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)
    trainer.save_model(args.output_dir)
    processor.save_pretrained(args.output_dir)

    run_config = {
        "seed": args.seed,
        "base_model": args.base_model,
        "preprocess": args.preprocess,
        "full_finetune": args.full_finetune,
        "lora_r": None if args.full_finetune else args.lora_r,
        "lora_alpha": None if args.full_finetune else (args.lora_alpha or 2 * args.lora_r),
        "lora_dropout": None if args.full_finetune else args.lora_dropout,
        "target_modules": None if args.full_finetune else args.target_modules,
        "epochs": args.epochs,
        "learning_rate": args.learning_rate,
        "max_target_length": args.max_target_length,
        "generation_max_length": generation_max_length,
        "batch_size": args.batch_size,
        "grad_accum": args.grad_accum,
        "eval_strategy": args.eval_strategy,
        "save_strategy": args.save_strategy,
        "eval_steps": args.eval_steps,
        "save_steps": args.save_steps,
        "synthetic_dir": args.synthetic_dir,
        "synthetic_limit": args.synthetic_limit,
        "resume_from_checkpoint": args.resume_from_checkpoint,
        "trainable_params": trainable_params,
        "total_params": total_params,
    }
    with open(os.path.join(args.output_dir, "run_config.json"), "w", encoding="utf-8") as fh:
        json.dump(run_config, fh, ensure_ascii=False, indent=2)
    print(f"Training complete. Saved {'model' if args.full_finetune else 'LoRA adapter'} to {args.output_dir}")

    if args.eval_test:
        print("Evaluating on the official test split...")
        if args.parquet_dir:
            test_dataset = ParquetDevanagariDataset(
                Path(args.parquet_dir).expanduser().resolve(),
                "test",
                processor,
                args.max_target_length,
                limit=args.eval_limit,
                preprocess=args.preprocess,
            )
        else:
            test_dataset = HuggingFaceDevanagariDataset(
                "test", processor, args.max_target_length,
                limit=args.eval_limit, preprocess=args.preprocess,
            )
        test_results = trainer.predict(test_dataset)
        test_metrics = {**test_results.metrics, "num_samples": len(test_dataset)}
        with open(os.path.join(args.output_dir, "test_metrics.json"), "w", encoding="utf-8") as fh:
            json.dump(test_metrics, fh, ensure_ascii=False, indent=2)
        print(f"Test metrics: {test_metrics}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper1/experiments/akshara.py
"""
Devanagari akshara (orthographic syllable / grapheme cluster) segmentation
and script-aware error categorization.

An akshara is the perceptual writing unit of Devanagari:
    (C halant)* C (nukta)? (matra)? (sign)*   e.g. क्ष्मी = क + ् + ष + ् + म + ी
    V (sign)*                                 independent vowel, e.g. आँ
Codepoint-level CER treats क्ष (3 codepoints) and क (1) asymmetrically;
akshara-level metrics weight them equally, matching how readers perceive errors.
"""

from __future__ import annotations

from dataclasses import dataclass

VIRAMA = "्"  # ्
NUKTA = "़"   # ़
ZWJ_ZWNJ = {"‌", "‍"}

# Unicode Devanagari block ranges
_CONSONANTS = set(
    [chr(c) for c in range(0x0915, 0x093A)]  # क..ह
    + [chr(c) for c in range(0x0958, 0x0960)]  # nukta consonants क़..य़
    + ["ॻ", "ॼ", "ॾ", "ॿ"]  # rare extensions
)
_INDEPENDENT_VOWELS = {chr(c) for c in range(0x0904, 0x0915)}  # ऄ..औ
_MATRAS = {chr(c) for c in range(0x093E, 0x094D)} | {"ॢ", "ॣ", "ऺ", "ऻ", "ॎ", "ॏ"}
_SIGNS = {"ँ", "ं", "ः"}  # candrabindu, anusvara, visarga
_DIGITS = {chr(c) for c in range(0x0966, 0x0970)}  # ०..९


def is_consonant(ch: str) -> bool:
    return ch in _CONSONANTS


def split_aksharas(text: str) -> list[str]:
    """Segment NFC-normalized Devanagari text into akshara clusters.
    Non-Devanagari characters become single-character clusters."""
    clusters: list[str] = []
    i, n = 0, len(text)
    while i < n:
        ch = text[i]
        if is_consonant(ch):
            j = i + 1
            while j < n and text[j] == NUKTA:
                j += 1
            # consume (halant [ZWJ] consonant)* chains — conjuncts
            while (
                j < n
                and text[j] == VIRAMA
                and (
                    (j + 1 < n and is_consonant(text[j + 1]))
                    or (j + 2 < n and text[j + 1] in ZWJ_ZWNJ and is_consonant(text[j + 2]))
                )
            ):
                j += 2 if is_consonant(text[j + 1]) else 3
                while j < n and text[j] == NUKTA:
                    j += 1
            # word-final dead consonant (trailing halant)
            if j < n and text[j] == VIRAMA and (j + 1 == n or not is_consonant(text[j + 1])):
                j += 1
            if j < n and text[j] in _MATRAS:
                j += 1
            while j < n and text[j] in _SIGNS:
                j += 1
            clusters.append(text[i:j])
            i = j
        elif ch in _INDEPENDENT_VOWELS:
            j = i + 1
            while j < n and text[j] in _SIGNS:
                j += 1
            clusters.append(text[i:j])
            i = j
        else:
            clusters.append(ch)
            i += 1
    return clusters


def is_conjunct(cluster: str) -> bool:
    """True if the cluster contains a consonant-joining virama (e.g. क्ष, त्र, स्थ)."""
    for k, ch in enumerate(cluster):
        if ch == VIRAMA and k + 1 < len(cluster):
            nxt = cluster[k + 1]
            if is_consonant(nxt) or nxt in ZWJ_ZWNJ:
                return True
    return False


def has_matra(cluster: str) -> bool:
    return any(ch in _MATRAS for ch in cluster)


def has_sign(cluster: str) -> bool:
    return any(ch in _SIGNS for ch in cluster)


def base_consonants(cluster: str) -> str:
    """The consonant/vowel skeleton of a cluster, stripped of matras and signs."""
    return "".join(ch for ch in cluster if is_consonant(ch) or ch in _INDEPENDENT_VOWELS or ch == VIRAMA)


@dataclass
class AksharaEdit:
    op: str  # "sub" | "ins" | "del"
    ref: str  # reference cluster ("" for insertions)
    pred: str  # predicted cluster ("" for deletions)

    def category(self) -> str:
        """Script-aware error category, checked most-specific first."""
        ref, pred = self.ref, self.pred
        if self.op == "ins":
            return "insertion_conjunct" if is_conjunct(pred) else "insertion"
        if self.op == "del":
            return "deletion_conjunct" if is_conjunct(ref) else "deletion"
        # substitution subtypes
        if is_conjunct(ref) or is_conjunct(pred):
            return "conjunct_substitution"
        if base_consonants(ref) == base_consonants(pred):
            if has_sign(ref) != has_sign(pred) and has_matra(ref) == has_matra(pred):
                return "sign_error"  # anusvara/candrabindu/visarga only
            return "matra_error"  # same skeleton, different vowel marking
        return "base_substitution"


def align_aksharas(ref_clusters: list[str], pred_clusters: list[str]) -> list[AksharaEdit]:
    """Levenshtein alignment over cluster sequences; returns only edit ops."""
    m, n = len(ref_clusters), len(pred_clusters)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_clusters[i - 1] == pred_clusters[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)

    edits: list[AksharaEdit] = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] and ref_clusters[i - 1] == pred_clusters[j - 1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            edits.append(AksharaEdit("sub", ref_clusters[i - 1], pred_clusters[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            edits.append(AksharaEdit("del", ref_clusters[i - 1], ""))
            i -= 1
        else:
            edits.append(AksharaEdit("ins", "", pred_clusters[j - 1]))
            j -= 1
    edits.reverse()
    return edits


In [ ]:
%%writefile paper1/experiments/common.py
"""
Shared utilities for Paper 1 experiments: seeding, model loading, and metrics.

All metrics operate on NFC-normalized Unicode. This matters for Devanagari:
the same visual word can be encoded with different codepoint sequences
(e.g. precomposed vs decomposed nukta forms), which silently inflates CER.
"""

from __future__ import annotations

import json
import os
import random
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import editdistance
import numpy as np
import torch
from peft import PeftModel
from transformers import AutoTokenizer, TrOCRProcessor, ViTImageProcessor, VisionEncoderDecoderModel

PROJECT_ROOT = Path(__file__).resolve().parents[2]
RESULTS_DIR = Path(__file__).resolve().parent / "results"
DATASET_NAME = "c3rl/IIIT-INDIC-HW-WORDS-Hindi"
DEFAULT_BASE_MODEL = "paudelanil/trocr-devanagari-2"
DEFAULT_IMAGE_PROCESSOR = "google/vit-base-patch16-224-in21k"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_best_torch_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFC", text.strip())


def cer(prediction: str, reference: str) -> float:
    """Character error rate over NFC-normalized codepoints."""
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)
    if not reference:
        return 0.0 if not prediction else 1.0
    return editdistance.eval(prediction, reference) / len(reference)


def word_error(prediction: str, reference: str) -> int:
    """Exact-match word error (0 = correct). Dataset is word-level, so
    aggregate word error rate == 1 - word recognition accuracy (WRA)."""
    return int(normalize_text(prediction) != normalize_text(reference))


def akshara_error_rate(prediction: str, reference: str) -> float:
    """Edit distance over akshara (grapheme cluster) sequences."""
    from paper1.experiments.akshara import split_aksharas

    pred_units = split_aksharas(normalize_text(prediction))
    ref_units = split_aksharas(normalize_text(reference))
    if not ref_units:
        return 0.0 if not pred_units else 1.0
    return editdistance.eval(pred_units, ref_units) / len(ref_units)


def bootstrap_ci(values: list[float], n_resamples: int = 1000, alpha: float = 0.05,
                 seed: int = 0) -> tuple[float, float]:
    """Percentile bootstrap CI for the mean of per-sample values."""
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=np.float64)
    means = np.empty(n_resamples)
    for i in range(n_resamples):
        means[i] = rng.choice(arr, size=len(arr), replace=True).mean()
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


@dataclass
class EvalSummary:
    run_name: str
    num_samples: int
    cer: float
    cer_ci95: tuple[float, float]
    aer: float  # akshara error rate
    word_accuracy: float
    trainable_params: Optional[int] = None
    total_params: Optional[int] = None
    config: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "run_name": self.run_name,
            "num_samples": self.num_samples,
            "cer": round(self.cer, 5),
            "cer_ci95": [round(v, 5) for v in self.cer_ci95],
            "aer": round(self.aer, 5),
            "word_accuracy": round(self.word_accuracy, 5),
            "trainable_params": self.trainable_params,
            "total_params": self.total_params,
            "config": self.config,
        }


def load_processor(model_name: str, processor_source: Optional[str] = None) -> TrOCRProcessor:
    """Mirror backend/trocr_engine.py processor resolution so eval matches serving."""
    if processor_source:
        try:
            image_processor = ViTImageProcessor.from_pretrained(processor_source)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            return TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
        except Exception:
            pass
    try:
        return TrOCRProcessor.from_pretrained(model_name)
    except Exception as exc:
        print(f"[common] Processor fallback for '{model_name}': {exc}")
        image_processor = ViTImageProcessor.from_pretrained(DEFAULT_IMAGE_PROCESSOR)
        image_processor.image_mean = [0.5, 0.5, 0.5]
        image_processor.image_std = [0.5, 0.5, 0.5]
        image_processor.rescale_factor = 1.0 / 255.0
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        return TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)


def load_model(
    base_model: str = DEFAULT_BASE_MODEL,
    adapter_path: Optional[str] = None,
    full_model_path: Optional[str] = None,
    device: Optional[str] = None,
) -> tuple[torch.nn.Module, TrOCRProcessor, str]:
    """Load one of: base model, base+LoRA adapter, or a fully fine-tuned checkpoint.

    Returns (model, processor, device). Model is in eval mode on the device.
    """
    device = device or os.getenv("TROCR_DEVICE") or get_best_torch_device()

    if full_model_path:
        model = VisionEncoderDecoderModel.from_pretrained(full_model_path)
        processor = load_processor(full_model_path)
    else:
        processor_source = None
        if adapter_path and (Path(adapter_path) / "preprocessor_config.json").exists():
            processor_source = adapter_path
        processor = load_processor(base_model, processor_source)
        model = VisionEncoderDecoderModel.from_pretrained(base_model)
        if adapter_path:
            model = PeftModel.from_pretrained(model, adapter_path)

    base = model.get_base_model() if isinstance(model, PeftModel) else model
    base.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    base.config.pad_token_id = processor.tokenizer.pad_token_id
    base.config.eos_token_id = processor.tokenizer.sep_token_id
    base.config.vocab_size = base.config.decoder.vocab_size
    base.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    base.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    base.generation_config.eos_token_id = processor.tokenizer.sep_token_id

    model.to(device)
    model.eval()
    return model, processor, device


def count_params(model: torch.nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    if trainable == 0 and isinstance(model, PeftModel):
        trainable = sum(
            p.numel()
            for name, p in model.named_parameters()
            if "lora_" in name or "modules_to_save" in name
        )
    return trainable, total


def save_results(payload: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)
    print(f"[common] Wrote {output_path}")


In [ ]:
%%writefile paper1/experiments/evaluate.py
"""
Evaluate a TrOCR checkpoint (base / base+LoRA / full fine-tune) on the
official IIIT-INDIC-HW-WORDS-Hindi test split.

Writes a results JSON with aggregate metrics (CER + 95% bootstrap CI, akshara
error rate, word accuracy) and per-sample predictions for error analysis.

Examples:
    # Unadapted base checkpoint
    python -m paper1.experiments.evaluate --run-name base_checkpoint

    # Shipped LoRA adapter
    python -m paper1.experiments.evaluate \
        --adapter-path trocr-devanagari-lora-hf --run-name lora_r16_legacy

    # Full fine-tune checkpoint
    python -m paper1.experiments.evaluate \
        --full-model-path ./trocr-devanagari-full --run-name full_ft
"""

from __future__ import annotations

import argparse
import io
import json
import sys
from pathlib import Path

import torch
from datasets import load_dataset
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from paper1.experiments.common import (  # noqa: E402
    DATASET_NAME,
    DEFAULT_BASE_MODEL,
    RESULTS_DIR,
    EvalSummary,
    akshara_error_rate,
    bootstrap_ci,
    cer,
    count_params,
    load_model,
    normalize_text,
    save_results,
    set_seed,
    word_error,
)


def coerce_to_pil(image_data) -> Image.Image:
    if isinstance(image_data, Image.Image):
        return image_data.convert("RGB")
    if isinstance(image_data, dict) and "bytes" in image_data:
        return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
    if isinstance(image_data, bytes):
        return Image.open(io.BytesIO(image_data)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image_data)!r}")


def load_training_config(*paths: str | None) -> dict:
    """Load the run_config.json saved by train_trocr.py, if this is a trained run."""
    for raw_path in paths:
        if not raw_path:
            continue
        config_path = Path(raw_path) / "run_config.json"
        if config_path.exists():
            with config_path.open("r", encoding="utf-8") as fh:
                return json.load(fh)
    return {}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate TrOCR on the official test split.")
    parser.add_argument("--base-model", default=DEFAULT_BASE_MODEL)
    parser.add_argument("--adapter-path", default=None, help="LoRA adapter directory.")
    parser.add_argument("--full-model-path", default=None, help="Fully fine-tuned model directory.")
    parser.add_argument("--parquet-dir", default=None,
                        help="Optional local HF parquet directory containing <split>-*.parquet files.")
    parser.add_argument("--split", default="test")
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--num-beams", type=int, default=4)
    parser.add_argument("--max-length", type=int, default=None,
                        help="Generation max length. Defaults to the checkpoint generation config.")
    parser.add_argument("--limit", type=int, default=None, help="Evaluate a subset (smoke test).")
    parser.add_argument("--device", default=None)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--run-name", required=True, help="Identifier used in the results filename.")
    parser.add_argument("--app-preprocess", action="store_true",
                        help="Apply the serving pipeline's crop/pad preprocessing before the "
                             "processor (measures deployment-pipeline effect; off = raw eval).")
    parser.add_argument("--output-dir", default=str(RESULTS_DIR))
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)

    model, processor, device = load_model(
        base_model=args.base_model,
        adapter_path=args.adapter_path,
        full_model_path=args.full_model_path,
        device=args.device,
    )
    trainable, total = count_params(model)
    training_config = load_training_config(args.adapter_path, args.full_model_path)
    if training_config:
        trainable = training_config.get("trainable_params", trainable)
        total = training_config.get("total_params", total)
    print(f"[evaluate] Model on {device} — trainable {trainable:,} / total {total:,}")

    split_spec = f"train[:{args.limit}]" if args.limit else "train"
    if args.parquet_dir:
        parquet_dir = Path(args.parquet_dir).expanduser().resolve()
        files = sorted(str(path) for path in parquet_dir.glob(f"{args.split}-*.parquet"))
        if not files:
            raise FileNotFoundError(f"No parquet files found for {args.split!r} in {parquet_dir}")
        dataset = load_dataset("parquet", data_files=files, split=split_spec)
    else:
        remote_split_spec = f"{args.split}[:{args.limit}]" if args.limit else args.split
        dataset = load_dataset(DATASET_NAME, split=remote_split_spec)
    print(f"[evaluate] {len(dataset)} samples from split '{args.split}'")
    max_length = args.max_length or getattr(model.generation_config, "max_length", None) or 128

    app_preprocess = None
    if args.app_preprocess:
        from backend.preprocessing import preprocess_pil_for_ocr as app_preprocess

    samples: list[dict] = []
    for start in range(0, len(dataset), args.batch_size):
        batch = dataset[start : start + args.batch_size]
        images = [coerce_to_pil(img) for img in batch["image"]]
        if app_preprocess:
            images = [app_preprocess(img) for img in images]
        references = [str(t) for t in batch["text"]]

        pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)
        with torch.inference_mode():
            output_ids = model.generate(
                inputs=pixel_values,
                max_length=max_length,
                num_beams=args.num_beams,
                early_stopping=True,
            )
        predictions = processor.batch_decode(output_ids, skip_special_tokens=True)

        for offset, (pred, ref) in enumerate(zip(predictions, references)):
            samples.append({
                "index": start + offset,
                "reference": normalize_text(ref),
                "prediction": normalize_text(pred),
                "cer": round(cer(pred, ref), 5),
                "aer": round(akshara_error_rate(pred, ref), 5),
                "word_error": word_error(pred, ref),
            })
        done = min(start + args.batch_size, len(dataset))
        if done % (args.batch_size * 20) < args.batch_size or done == len(dataset):
            running_cer = sum(s["cer"] for s in samples) / len(samples)
            print(f"[evaluate] {done}/{len(dataset)}  running CER={running_cer:.4f}")

    cers = [s["cer"] for s in samples]
    summary = EvalSummary(
        run_name=args.run_name,
        num_samples=len(samples),
        cer=sum(cers) / len(cers),
        cer_ci95=bootstrap_ci(cers, seed=args.seed),
        aer=sum(s["aer"] for s in samples) / len(samples),
        word_accuracy=1.0 - sum(s["word_error"] for s in samples) / len(samples),
        trainable_params=trainable,
        total_params=total,
        config={
            "base_model": args.base_model,
            "adapter_path": args.adapter_path,
            "full_model_path": args.full_model_path,
            "split": args.split,
            "num_beams": args.num_beams,
            "max_length": max_length,
            "app_preprocess": args.app_preprocess,
            "seed": args.seed,
            "device": device,
            "limit": args.limit,
            "training_config": training_config,
        },
    )

    print(f"\n[evaluate] {args.run_name}")
    print(f"  CER            {summary.cer:.4f}  (95% CI {summary.cer_ci95[0]:.4f}–{summary.cer_ci95[1]:.4f})")
    print(f"  Akshara ER     {summary.aer:.4f}")
    print(f"  Word accuracy  {summary.word_accuracy:.4f}")

    output_path = Path(args.output_dir) / f"{args.run_name}.json"
    save_results({"summary": summary.to_dict(), "samples": samples}, output_path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper1/experiments/error_analysis.py
"""
Script-aware error analysis over a results JSON produced by evaluate.py.

Answers the Devanagari-specific questions no Latin-script HTR paper covers:
  - How much error comes from conjunct clusters vs simple aksharas?
  - Are matras (dependent vowels) or nasalization signs the dominant failure?
  - Which akshara confusion pairs are most frequent?
  - Does error grow with word length / conjunct density?

Usage:
    python -m paper1.experiments.error_analysis \
        --results paper1/experiments/results/lora_r16_legacy.json
"""

from __future__ import annotations

import argparse
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from paper1.experiments.akshara import (  # noqa: E402
    align_aksharas,
    is_conjunct,
    split_aksharas,
)

CATEGORY_LABELS = {
    "conjunct_substitution": "Conjunct substitution",
    "matra_error": "Matra (dependent vowel) error",
    "sign_error": "Nasalization/visarga sign error",
    "base_substitution": "Base consonant substitution",
    "insertion": "Insertion",
    "insertion_conjunct": "Insertion (conjunct)",
    "deletion": "Deletion",
    "deletion_conjunct": "Deletion (conjunct)",
}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Devanagari error analysis of eval results.")
    parser.add_argument("--results", required=True, help="Path to evaluate.py results JSON.")
    parser.add_argument("--top-confusions", type=int, default=25)
    parser.add_argument("--output", default=None,
                        help="Output markdown path (default: <results>_analysis.md).")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    results_path = Path(args.results)
    with results_path.open("r", encoding="utf-8") as fh:
        data = json.load(fh)

    samples = data["samples"]
    run_name = data["summary"]["run_name"]

    category_counts: Counter[str] = Counter()
    confusion_pairs: Counter[tuple[str, str]] = Counter()
    total_ref_aksharas = 0
    conjunct_ref_total = 0
    conjunct_ref_errors = 0
    simple_ref_total = 0
    simple_ref_errors = 0
    # error rate bucketed by word length (in aksharas) and conjunct density
    length_buckets: dict[str, list[float]] = {"1-2": [], "3-4": [], "5-6": [], "7+": []}
    conjunct_buckets: dict[str, list[float]] = {"0 conjuncts": [], "1 conjunct": [], "2+ conjuncts": []}

    for sample in samples:
        ref_clusters = split_aksharas(sample["reference"])
        pred_clusters = split_aksharas(sample["prediction"])
        edits = align_aksharas(ref_clusters, pred_clusters)

        total_ref_aksharas += len(ref_clusters)
        n_conjuncts = sum(1 for c in ref_clusters if is_conjunct(c))
        conjunct_ref_total += n_conjuncts
        simple_ref_total += len(ref_clusters) - n_conjuncts

        for edit in edits:
            category_counts[edit.category()] += 1
            if edit.ref:
                if is_conjunct(edit.ref):
                    conjunct_ref_errors += 1
                else:
                    simple_ref_errors += 1
            if edit.op == "sub":
                confusion_pairs[(edit.ref, edit.pred)] += 1

        n = len(ref_clusters)
        length_key = "1-2" if n <= 2 else "3-4" if n <= 4 else "5-6" if n <= 6 else "7+"
        length_buckets[length_key].append(sample["aer"])
        conjunct_key = "0 conjuncts" if n_conjuncts == 0 else "1 conjunct" if n_conjuncts == 1 else "2+ conjuncts"
        conjunct_buckets[conjunct_key].append(sample["aer"])

    total_edits = sum(category_counts.values())

    lines: list[str] = []
    lines.append(f"# Devanagari error analysis — `{run_name}`")
    lines.append("")
    lines.append(f"Samples: {len(samples)} — total reference aksharas: {total_ref_aksharas} — "
                 f"total akshara-level edits: {total_edits}")
    lines.append("")

    lines.append("## Error category breakdown")
    lines.append("")
    lines.append("| Category | Count | % of errors |")
    lines.append("|---|---:|---:|")
    for key, count in category_counts.most_common():
        share = 100.0 * count / total_edits if total_edits else 0.0
        lines.append(f"| {CATEGORY_LABELS.get(key, key)} | {count} | {share:.1f}% |")
    lines.append("")

    lines.append("## Conjunct vs simple akshara error rate")
    lines.append("")
    lines.append("| Akshara type | Occurrences | Errored | Error rate |")
    lines.append("|---|---:|---:|---:|")
    conj_rate = conjunct_ref_errors / conjunct_ref_total if conjunct_ref_total else 0.0
    simple_rate = simple_ref_errors / simple_ref_total if simple_ref_total else 0.0
    lines.append(f"| Conjunct (contains virama-joined consonants) | {conjunct_ref_total} | {conjunct_ref_errors} | {conj_rate:.4f} |")
    lines.append(f"| Simple | {simple_ref_total} | {simple_ref_errors} | {simple_rate:.4f} |")
    lines.append("")

    lines.append("## Top akshara confusion pairs (reference → prediction)")
    lines.append("")
    lines.append("| Reference | Prediction | Count |")
    lines.append("|---|---|---:|")
    for (ref, pred), count in confusion_pairs.most_common(args.top_confusions):
        lines.append(f"| {ref} | {pred} | {count} |")
    lines.append("")

    lines.append("## Akshara error rate by word length (aksharas)")
    lines.append("")
    lines.append("| Length | Words | Mean AER |")
    lines.append("|---|---:|---:|")
    for key, values in length_buckets.items():
        mean = sum(values) / len(values) if values else 0.0
        lines.append(f"| {key} | {len(values)} | {mean:.4f} |")
    lines.append("")

    lines.append("## Akshara error rate by conjunct count")
    lines.append("")
    lines.append("| Conjuncts in word | Words | Mean AER |")
    lines.append("|---|---:|---:|")
    for key, values in conjunct_buckets.items():
        mean = sum(values) / len(values) if values else 0.0
        lines.append(f"| {key} | {len(values)} | {mean:.4f} |")
    lines.append("")

    output_path = Path(args.output) if args.output else results_path.with_name(
        results_path.stem + "_analysis.md"
    )
    output_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"[error_analysis] Wrote {output_path}")
    print("\n".join(lines[:20]))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paper1/experiments/make_tables.py
"""
Aggregate results/*.json from evaluate.py into publication tables
(Markdown for inspection, LaTeX booktabs for the paper).

Grouping is by run-name convention:
    base_checkpoint, full_ft, lora_r{R}_{modules}     -> main + ablation tables
    lora_r16_legacy_synth{PCT}                        -> augmentation table

Usage:
    python -m paper1.experiments.make_tables
"""

from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

RESULTS_DIR = Path(__file__).resolve().parent / "results"
REPO_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_PAPER_TABLE = REPO_ROOT / "paper1" / "paper" / "tables" / "tables.tex"


def load_summaries(results_dir: Path) -> list[dict]:
    summaries = []
    for path in sorted(results_dir.glob("*.json")):
        with path.open("r", encoding="utf-8") as fh:
            data = json.load(fh)
        if "summary" in data:
            summaries.append(data["summary"])
    return summaries


def fmt_params(count) -> str:
    if count is None:
        return "—"
    return f"{count / 1e6:.2f}M"


def fmt_ci(ci: list[float]) -> str:
    return f"[{ci[0]:.4f}, {ci[1]:.4f}]"


def markdown_table(summaries: list[dict]) -> str:
    lines = [
        "| Run | Trainable params | CER ↓ | CER 95% CI | Akshara ER ↓ | Word acc. ↑ |",
        "|---|---:|---:|---|---:|---:|",
    ]
    for s in summaries:
        lines.append(
            f"| {s['run_name']} | {fmt_params(s.get('trainable_params'))} | "
            f"{s['cer']:.4f} | {fmt_ci(s['cer_ci95'])} | {s['aer']:.4f} | {s['word_accuracy']:.4f} |"
        )
    return "\n".join(lines)


def latex_table(summaries: list[dict], caption: str, label: str) -> str:
    rows = []
    for s in summaries:
        name = s["run_name"].replace("_", r"\_")
        rows.append(
            f"    {name} & {fmt_params(s.get('trainable_params'))} & "
            f"{s['cer']:.4f} & {s['aer']:.4f} & {s['word_accuracy']:.4f} \\\\"
        )
    body = "\n".join(rows)
    return (
        "\\begin{table}[t]\n"
        "  \\centering\n"
        f"  \\caption{{{caption}}}\n"
        f"  \\label{{{label}}}\n"
        "  \\begin{tabular}{lrrrr}\n"
        "    \\toprule\n"
        "    Run & Trainable & CER $\\downarrow$ & AER $\\downarrow$ & WAcc $\\uparrow$ \\\\\n"
        "    \\midrule\n"
        f"{body}\n"
        "    \\bottomrule\n"
        "  \\end{tabular}\n"
        "\\end{table}\n"
    )


def main() -> None:
    parser = argparse.ArgumentParser(description="Build result tables from eval JSONs.")
    parser.add_argument("--results-dir", default=str(RESULTS_DIR))
    parser.add_argument("--output-dir", default=str(RESULTS_DIR / "tables"))
    parser.add_argument(
        "--paper-table-path",
        default=str(DEFAULT_PAPER_TABLE),
        help="Optional LaTeX table mirror used by paper1/paper/main.tex; pass '' to skip.",
    )
    args = parser.parse_args()

    results_dir = Path(args.results_dir)
    summaries = load_summaries(results_dir)
    if not summaries:
        print(f"[make_tables] No results in {results_dir} — run evaluate.py first.")
        return

    synth = [s for s in summaries if re.search(r"synth", s["run_name"])]
    main_runs = [s for s in summaries if s not in synth]
    main_runs.sort(key=lambda s: s["cer"])
    synth.sort(key=lambda s: s["run_name"])

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    md_parts = ["# Paper 1 result tables", "", "## Main comparison / LoRA ablation", "",
                markdown_table(main_runs), ""]
    tex_parts = [latex_table(
        main_runs,
        "Test-split results on IIIT-INDIC-HW-WORDS-Hindi. CER: character error rate "
        "(NFC codepoints); AER: akshara error rate; WAcc: word recognition accuracy.",
        "tab:main",
    )]
    if synth:
        md_parts += ["## Synthetic data augmentation", "", markdown_table(synth), ""]
        tex_parts.append(latex_table(
            synth,
            "Effect of LDM-generated synthetic training data on recognition.",
            "tab:synth",
        ))

    md_text = "\n".join(md_parts)
    tex_text = "\n\n".join(tex_parts)
    (output_dir / "tables.md").write_text(md_text, encoding="utf-8")
    (output_dir / "tables.tex").write_text(tex_text, encoding="utf-8")
    if args.paper_table_path:
        paper_table_path = Path(args.paper_table_path)
        paper_table_path.parent.mkdir(parents=True, exist_ok=True)
        paper_table_path.write_text(tex_text, encoding="utf-8")
    print(f"[make_tables] Wrote {output_dir}/tables.md and tables.tex "
          f"({len(main_runs)} main runs, {len(synth)} augmentation runs)")
    if args.paper_table_path:
        print(f"[make_tables] Mirrored LaTeX table to {args.paper_table_path}")
    print()
    print(md_text)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Sanity: the transformers double-shift loss bug must be bypassed ----
import torch, io, sys
import torch.nn.functional as F
sys.path.insert(0, ".")
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from datasets import load_dataset
from PIL import Image
from backend.preprocessing import preprocess_pil_for_ocr

m = VisionEncoderDecoderModel.from_pretrained("paudelanil/trocr-devanagari-2").eval()
p = TrOCRProcessor.from_pretrained("paudelanil/trocr-devanagari-2")
ds = load_dataset("parquet", data_files=["data/iiit_hindi_parquet/train-00000-of-00003.parquet"], split="train[:8]")
pv, lb = [], []
for s in ds:
    img = s["image"]
    if isinstance(img, dict): img = Image.open(io.BytesIO(img["bytes"]))
    pv.append(p(images=preprocess_pil_for_ocr(img.convert("RGB")), return_tensors="pt").pixel_values[0])
    ids = p.tokenizer(str(s["text"]), padding="max_length", max_length=32, truncation=True).input_ids
    lb.append(torch.tensor([i if i != 1 else -100 for i in ids]))
pv, lb = torch.stack(pv), torch.stack(lb)
sh = lb.new_full(lb.shape, 1); sh[:,1:] = lb[:,:-1].clone(); sh[:,0] = 0; sh[sh==-100] = 1
with torch.no_grad():
    manual = F.cross_entropy(m(pixel_values=pv, decoder_input_ids=sh).logits.reshape(-1, m.config.decoder.vocab_size), lb.reshape(-1), ignore_index=-100)
    internal = m(pixel_values=pv, labels=lb).loss
print(f"manual(correct): {manual.item():.3f}   internal(buggy): {internal.item():.3f}")
assert manual.item() < 2.0, "alignment broken — do not train!"
print("OK — training uses the manual path.")

In [ ]:
# ==================== CONFIG ====================
EPOCHS = 2
SEED = 42
BASE_MODEL = "paudelanil/trocr-devanagari-2"
PARQUET = "data/iiit_hindi_parquet"
OUT = "paper1/runs"

# (name, extra args) — comment out any already finished
RUNS = [
    ("lora_r16_attn",     "--lora-r 16 --target-modules attn"),
    ("lora_r16_legacy",   "--lora-r 16 --target-modules legacy"),
    ("lora_r4_attn",      "--lora-r 4  --target-modules attn"),
    ("lora_r8_attn",      "--lora-r 8  --target-modules attn"),
    ("lora_r32_attn",     "--lora-r 32 --target-modules attn"),
    ("lora_r16_attn_ffn", "--lora-r 16 --target-modules attn-ffn"),
    ("full_ft",           "--full-finetune --learning-rate 2e-5"),
]

In [ ]:
# ==================== TRAIN + EVAL ALL ====================
import os, subprocess, time

def sh(cmd):
    print(">>", cmd, flush=True)
    return subprocess.call(cmd, shell=True)

if not os.path.exists("paper1/experiments/results/base_prep.json"):
    sh(f"python -m paper1.experiments.evaluate --run-name base_prep "
       f"--parquet-dir {PARQUET} --base-model {BASE_MODEL} "
       f"--batch-size 32 --max-length 32 --app-preprocess")

for name, extra in RUNS:
    t0 = time.time()
    epochs = 1 if name == "full_ft" else EPOCHS
    bs = 8 if name == "full_ft" else 16
    if os.path.exists(f"{OUT}/{name}/run_config.json"):
        print(f"=== {name}: already trained, skip ===")
    else:
        rc = sh(f"python backend/train_trocr.py "
                f"--parquet-dir {PARQUET} --base-model {BASE_MODEL} "
                f"--output-dir {OUT}/{name} --epochs {epochs} --seed {SEED} "
                f"--batch-size {bs} --eval-batch-size 32 --grad-accum 1 "
                f"--eval-strategy epoch --save-strategy epoch --eval-limit 2000 "
                f"--generation-max-length 32 --preprocess app --num-workers 2 {extra}")
        if rc != 0:
            print(f"!!! {name} TRAINING FAILED — continuing with next run"); continue
    if not os.path.exists(f"paper1/experiments/results/{name}.json"):
        model_arg = (f"--full-model-path {OUT}/{name}" if name == "full_ft"
                     else f"--adapter-path {OUT}/{name}")
        sh(f"python -m paper1.experiments.evaluate --run-name {name} "
           f"--parquet-dir {PARQUET} --base-model {BASE_MODEL} {model_arg} "
           f"--batch-size 32 --max-length 32 --app-preprocess")
        sh(f"python -m paper1.experiments.error_analysis "
           f"--results paper1/experiments/results/{name}.json")
    print(f"=== {name} complete in {(time.time()-t0)/60:.0f} min ===")

sh("python -m paper1.experiments.make_tables")

In [ ]:
# ==================== RESULTS SUMMARY ====================
import json, glob
rows = []
for f in sorted(glob.glob("paper1/experiments/results/*.json")):
    if "audit" in f: continue
    s = json.load(open(f))["summary"]
    tp = s.get("trainable_params")
    rows.append((s["run_name"], f"{tp/1e6:.2f}M" if tp else "-", s["cer"], s["aer"], s["word_accuracy"]))
rows.sort(key=lambda r: r[2])
print(f"{'run':22} {'params':>8} {'CER':>7} {'AER':>7} {'WAcc':>7}")
for r in rows:
    print(f"{r[0]:22} {r[1]:>8} {r[2]:>7.4f} {r[3]:>7.4f} {r[4]:>7.4f}")

In [ ]:
# ==================== PACKAGE FOR DOWNLOAD ====================
!rm -f /kaggle/working/paper1_artifacts.zip
!zip -qr /kaggle/working/paper1_artifacts.zip \
    paper1/experiments/results \
    paper1/runs/*/adapter_model.safetensors \
    paper1/runs/*/adapter_config.json \
    paper1/runs/*/run_config.json \
    paper1/runs/*/test_metrics.json 2>/dev/null
!zip -qr /kaggle/working/full_ft_model.zip paper1/runs/full_ft/model.safetensors paper1/runs/full_ft/config.json 2>/dev/null || echo "no full_ft weights"
!ls -la /kaggle/working/*.zip
print("Download paper1_artifacts.zip, unzip into the DevGen repo root — paths land in place.")